In [1]:
import geopandas as gpd
import os
import requests
from shapely.validation import make_valid

# Cargar el límite oficial de Usaquén
usaquen_oficial = gpd.read_file("../data/raw/limites/loca.json")

usaquen_filtrado = usaquen_oficial[
    usaquen_oficial["LocNombre"] == "USAQUEN"
]

usaquen_filtrado = usaquen_filtrado.to_crs(epsg=4326)

usaquen_polygon = usaquen_filtrado.geometry.iloc[0]

if not usaquen_polygon.is_valid:
    usaquen_polygon = make_valid(usaquen_polygon)

print(f"Polígono válido: {usaquen_polygon.is_valid}")
print(usaquen_polygon.geom_type)

Polígono válido: True
Polygon


In [2]:
coords = list(usaquen_polygon.exterior.coords)

poly_coords = " ".join(
    f"{lat} {lon}"
    for lon, lat in coords
)

print(poly_coords[:500])

4.6645853120000424 -74.01116193799993 4.664600325000038 -74.01116635999995 4.664670392000062 -74.0111970449999 4.6647556270000905 -74.0112404919999 4.664802361000056 -74.01126767399995 4.664843971000039 -74.01128191799995 4.664872431000049 -74.01129356699994 4.664905297000075 -74.01130481899992 4.664934909000067 -74.01131778699994 4.664984942000046 -74.01134688799993 4.665016592000086 -74.0113534969999 4.665055760000087 -74.01137346099995 4.665099846000089 -74.01138855699992 4.665135217000056 -7


In [3]:
ruta_archivo = '../data/processed/usaquen.osm.xml'

query = f"""
[out:xml][timeout:180];
(
  way["highway"](poly:"{poly_coords}");
);
(._;>;);
out meta;
"""

response = requests.post(
    "https://overpass-api.de/api/interpreter",
    data={'data': query.encode('utf-8')},
    headers={"User-Agent": "MiAplicacionGeografica/1.0"},
    timeout=300
)

if response.status_code == 200:
    #Guarda el contenido en disco
    with open(ruta_archivo, 'wb') as f:
        f.write(response.content)
    
    print(f"Archivo guardado en: {os.path.abspath(ruta_archivo)}\n")

    #primeras líneas
    print("Contenido del archivo")
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        for i in range(10):
            print(f.readline(), end='')
else:
    print(f"Error HTTP {response.status_code}: {response.text[:200]}")

Archivo guardado en: c:\Users\USUARIO\Downloads\simulacion-movilidad-urbana\data\processed\usaquen.osm.xml

Contenido del archivo
<?xml version="1.0" encoding="UTF-8"?>
<osm version="0.6" generator="Overpass API 0.7.62.11 87bfad18">
<note>The data included in this document is from www.openstreetmap.org. The data is made available under ODbL.</note>
<meta osm_base="2026-08-21T18:47:06Z"/>

  <node id="253845462" lat="4.6752483" lon="-74.0243597" version="18" timestamp="2026-01-01T21:02:09Z" changeset="176719144" uid="1759764" user="Facalderonm"/>
  <node id="253845500" lat="4.6752084" lon="-74.0244588" version="12" timestamp="2026-01-01T21:02:09Z" changeset="176719144" uid="1759764" user="Facalderonm"/>
  <node id="253845875" lat="4.6757979" lon="-74.0264972" version="18" timestamp="2026-01-01T21:02:09Z" changeset="176719144" uid="1759764" user="Facalderonm"/>
  <node id="253846345" lat="4.6680374" lon="-74.0124907" version="9" timestamp="2020-06-01T15:55:02Z" changeset="86053783" uid="

In [10]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

OSM_FILE = "../data/processed/usaquen.osm.xml"
NET_FILE = "../data/processed/usaquen.net.xml"

In [11]:
tree_osm = ET.parse(OSM_FILE)
root_osm = tree_osm.getroot()

ways = []

for way in root_osm.findall("way"):
    tags = {
        tag.attrib["k"]: tag.attrib["v"]
        for tag in way.findall("tag")
    }

    if "highway" in tags:
        ways.append({
            "id": way.attrib["id"],
            **tags
        })

osm = pd.DataFrame(ways)

print(f"Total de vías OSM: {len(osm):,}")

print("\nTipos de highway:")
print(osm["highway"].value_counts())

Total de vías OSM: 12,310

Tipos de highway:
highway
footway           3160
residential       3039
service           2724
primary            972
secondary          657
tertiary           386
cycleway           257
steps              163
primary_link       144
pedestrian         142
trunk              141
proposed           124
path                86
corridor            81
construction        74
trunk_link          58
track               32
secondary_link      24
bus_stop            20
unclassified        16
tertiary_link        6
bridleway            2
services             1
platform             1
Name: count, dtype: int64


In [12]:
HIGHWAYS_VEHICULARES = [
    "motorway",
    "motorway_link",
    "trunk",
    "trunk_link",
    "primary",
    "primary_link",
    "secondary",
    "secondary_link",
    "tertiary",
    "tertiary_link",
    "unclassified",
    "residential",
    "living_street",
    "service"
]

osm_veh = osm[
    osm["highway"].isin(HIGHWAYS_VEHICULARES)
].copy()

print(f"Vías vehiculares: {len(osm_veh):,}")

Vías vehiculares: 8,167


In [24]:
total_veh = len(osm_veh)

sin_maxspeed = osm_veh["maxspeed"].isna().sum()
sin_lanes = osm_veh["lanes"].isna().sum()

print("=== ATRIBUTOS FALTANTES EN VÍAS VEHICULARES ===")

print(
    f"Sin maxspeed: {sin_maxspeed:,} "
    f"({sin_maxspeed / total_veh * 100:.2f}%)"
)

print(
    f"Sin lanes: {sin_lanes:,} "
    f"({sin_lanes / total_veh * 100:.2f}%)"
)

=== ATRIBUTOS FALTANTES EN VÍAS VEHICULARES ===
Sin maxspeed: 5,255 (64.34%)
Sin lanes: 4,034 (49.39%)


In [14]:
print("\n=== VALORES DE MAXSPEED ===")
print(osm_veh["maxspeed"].value_counts(dropna=False))

print("\n=== VALORES DE LANES ===")
print(osm_veh["lanes"].value_counts(dropna=False))


=== VALORES DE MAXSPEED ===
maxspeed
NaN    5255
30     2498
50       94
40       94
10       93
20       69
60       64
Name: count, dtype: int64

=== VALORES DE LANES ===
lanes
NaN    4034
2      3027
3       559
1       462
4        68
5        13
6         3
20        1
Name: count, dtype: int64


In [15]:
osm_veh["maxspeed_num"] = pd.to_numeric(
    osm_veh["maxspeed"],
    errors="coerce"
)

osm_veh["lanes_num"] = pd.to_numeric(
    osm_veh["lanes"],
    errors="coerce"
)

analisis_faltantes = (
    osm_veh
    .groupby("highway")
    .agg(
        tramos=("id", "count"),
        sin_maxspeed=("maxspeed_num", lambda x: x.isna().sum()),
        sin_lanes=("lanes_num", lambda x: x.isna().sum())
    )
)

analisis_faltantes["pct_sin_maxspeed"] = (
    analisis_faltantes["sin_maxspeed"] /
    analisis_faltantes["tramos"] * 100
)

analisis_faltantes["pct_sin_lanes"] = (
    analisis_faltantes["sin_lanes"] /
    analisis_faltantes["tramos"] * 100
)

analisis_faltantes.round(2)

,tramos,sin_maxspeed,sin_lanes,pct_sin_maxspeed,pct_sin_lanes
highway,,,,,
primary,972,574,0,59.05,0.00
primary_link,144,120,7,83.33,4.86
residential,3039,1431,1413,47.09,46.50
secondary,657,261,2,39.73,0.30
secondary_link,24,17,1,70.83,4.17
service,2724,2588,2588,95.01,95.01
tertiary,386,141,5,36.53,1.30
tertiary_link,6,5,2,83.33,33.33
trunk,141,71,0,50.35,0.00


In [16]:
tree_net = ET.parse(NET_FILE)
root_net = tree_net.getroot()

edges = []

for edge in root_net.findall("edge"):

    if edge.attrib.get("function") == "internal":
        continue

    lanes = edge.findall("lane")

    edges.append({
        "id": edge.attrib["id"],
        "type": edge.attrib.get("type"),
        "num_lanes": len(lanes),
        "speed": [
            float(lane.attrib["speed"])
            for lane in lanes
        ]
    })

net_edges = pd.DataFrame(edges)

print(f"Edges SUMO: {len(net_edges):,}")
print(f"Debería coincidir aproximadamente con NetEdit: 28,070")

Edges SUMO: 28,070
Debería coincidir aproximadamente con NetEdit: 28,070


In [17]:
net_edges["speed_mean"] = net_edges["speed"].apply(np.mean)
net_edges["speed_kmh"] = net_edges["speed_mean"] * 3.6

print("=== VELOCIDADES FINALES ===")

print(
    net_edges["speed_kmh"].describe()
)

print("\n=== CARRILES POR EDGE ===")

print(
    net_edges["num_lanes"].value_counts().sort_index()
)

=== VELOCIDADES FINALES ===
count    28070.000000
mean        28.376308
std         21.106055
min          5.004000
25%         10.008000
50%         20.016000
75%         29.988000
max        100.008000
Name: speed_kmh, dtype: float64

=== CARRILES POR EDGE ===
num_lanes
1     25174
2      2022
3       754
4        89
5        19
6         9
7         1
10        2
Name: count, dtype: int64


In [18]:
resumen_tipos = (
    net_edges
    .groupby("type")
    .agg(
        edges=("id", "count"),
        velocidad_promedio_kmh=("speed_kmh", "mean"),
        carriles_promedio=("num_lanes", "mean")
    )
    .sort_values("edges", ascending=False)
)

resumen_tipos.round(2)

,edges,velocidad_promedio_kmh,carriles_promedio
type,,,
highway.residential,8727,39.12,1.06
highway.footway,7367,10.03,1.00
highway.service,5870,19.83,1.00
highway.primary,1326,66.00,2.42
highway.cycleway,1187,20.02,1.00
highway.secondary,985,56.10,1.79
highway.tertiary,968,45.67,1.17
highway.pedestrian,368,10.01,1.02
highway.path,276,20.02,1.00


In [19]:
def obtener_permisos(lane):
    return {
        "allow": lane.attrib.get("allow"),
        "disallow": lane.attrib.get("disallow")
    }


allow_disallow = []

for edge in root_net.findall("edge"):

    if edge.attrib.get("function") == "internal":
        continue

    for lane in edge.findall("lane"):
        allow_disallow.append({
            "edge_id": edge.attrib["id"],
            "type": edge.attrib.get("type"),
            "allow": lane.attrib.get("allow"),
            "disallow": lane.attrib.get("disallow")
        })

permisos = pd.DataFrame(allow_disallow)

print("=== ALLOW ===")
print(permisos["allow"].value_counts(dropna=False).head(20))

print("\n=== DISALLOW ===")
print(permisos["disallow"].value_counts(dropna=False).head(20))

=== ALLOW ===
allow
NaN                                    16376
pedestrian                              7986
pedestrian delivery bicycle             5810
pedestrian bicycle                      1062
bicycle                                  427
pedestrian motorcycle moped bicycle      158
bus bicycle                              155
emergency authority bus bicycle           24
delivery bicycle                          14
Name: count, dtype: int64

=== DISALLOW ===
disallow
NaN                                                                                                                                  15636
tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone                       14711
pedestrian tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone              828
pedestrian bicycle tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft 

In [20]:
crossings = []

for node in root_osm.findall("node"):

    tags = {
        tag.attrib["k"]: tag.attrib["v"]
        for tag in node.findall("tag")
    }

    if tags.get("highway") == "crossing":
        crossings.append({
            "id": node.attrib["id"],
            **tags
        })

print(f"Cruces peatonales registrados en OSM: {len(crossings):,}")

Cruces peatonales registrados en OSM: 2,134


In [21]:
sidewalk_cols = [
    col for col in osm.columns
    if col.startswith("sidewalk")
]

print("Atributos de acera encontrados:")

for col in sidewalk_cols:
    print(f"\n{col}")
    print(osm[col].value_counts(dropna=False).head(10))

Atributos de acera encontrados:

sidewalk
sidewalk
NaN         11936
separate      314
both           38
right          14
no              8
Name: count, dtype: int64

sidewalk:right
sidewalk:right
NaN         12298
separate       10
no              2
Name: count, dtype: int64

sidewalk:left
sidewalk:left
NaN         12299
no              9
separate        2
Name: count, dtype: int64

sidewalk:both
sidewalk:both
NaN    12309
yes        1
Name: count, dtype: int64


In [22]:
print("=" * 50)
print("VALIDACIÓN FINAL DE LA RED")
print("=" * 50)

print(f"Nodos cargados en NetEdit: 12,989")
print(f"Edges SUMO: {len(net_edges):,}")
print(f"Tipos de vía SUMO: {net_edges['type'].nunique()}")
print(f"Cruces OSM registrados: {len(crossings):,}")

print("\nTipos de vía:")
print(net_edges["type"].value_counts())

print("\nLa red fue revisada visualmente en NetEdit.")

VALIDACIÓN FINAL DE LA RED
Nodos cargados en NetEdit: 12,989
Edges SUMO: 28,070
Tipos de vía SUMO: 19
Cruces OSM registrados: 2,134

Tipos de vía:
type
highway.residential       8727
highway.footway           7367
highway.service           5870
highway.primary           1326
highway.cycleway          1187
highway.secondary          985
highway.tertiary           968
highway.pedestrian         368
highway.path               276
highway.steps              237
highway.trunk              235
highway.primary_link       164
highway.track              162
highway.trunk_link          95
highway.unclassified        77
highway.secondary_link      18
highway.bridleway            4
highway.tertiary_link        3
highway.service|psv          1
Name: count, dtype: int64

La red fue revisada visualmente en NetEdit.


# Conversión de red OSM a SUMO y preparación multimodal

**Responsable:** Santiago Chitiva  
**Fase:** 2 - Preparación de datos (ETL)  
**Semana:** 3 
**Prerrequisito:** F1.3 - Red vial OSM validada (cerrado)

## Objetivo

Convertir la red vial de OpenStreetMap (OSM) correspondiente a la localidad de Usaquén al formato nativo de SUMO (`.net.xml`), ampliando el alcance de la red para considerar vehículos, bicicletas y peatones.

## Actividades realizadas

### 1. Revisión de la delimitación espacial

Se utilizó el límite oficial de la localidad de Usaquén empleado en F1.3, obtenido desde `loca.json` y filtrado mediante el atributo `LocNombre = 'USAQUEN'`.

El polígono fue transformado al sistema de coordenadas WGS84 (EPSG:4326) y se validó su geometría antes de utilizarlo para la consulta de OpenStreetMap.

### 2. Prueba exploratoria de la red con OSMnx

Inicialmente se utilizó OSMnx para consultar la red mediante `network_type="all"` y se comparó este resultado con un `custom_filter` que incluía categorías vehiculares y no motorizadas.

Se verificó que `network_type="all"` incorpora infraestructura relevante para el alcance multimodal, incluyendo:

- `footway`
- `cycleway`
- `path`
- `pedestrian`
- `steps`
- `corridor`
- `track`

además de las categorías de vías vehiculares convencionales.

### 3. Revisión de las categorías `highway`

Durante la comparación se identificó que OSMnx puede representar determinadas aristas con múltiples valores de `highway`, por ejemplo:

- `[residential, footway]`
- `[footway, steps]`
- `[residential, service]`
- `[footway, corridor]`

Se comprobó que estas combinaciones están relacionadas con la simplificación del grafo realizada por OSMnx. Al utilizar `simplify=False`, se conservaron los segmentos individuales y dejaron de aparecer las combinaciones de categorías como representación de una misma arista.

Sin embargo, esta configuración produjo una red considerablemente más fragmentada, por lo que no se utilizó como mecanismo definitivo para generar la red de SUMO.

### 4. Prueba inicial de conversión mediante OSMnx

Se realizó una primera prueba exportando el grafo de OSMnx mediante `ox.save_graph_xml()` y posteriormente convirtiéndolo a SUMO mediante `netconvert`.

La inspección del resultado en NetEdit mostró una representación diferente a la observada directamente en OSMnx, con numerosos segmentos que visualmente parecían desconectados.

Se verificó posteriormente la conectividad del grafo en OSMnx y se encontró que todos los nodos pertenecían a un único componente conectado, por lo que el problema visual no correspondía a una desconexión real del grafo.

### 5. Descarga directa de OSM mediante Overpass

Para evitar reconstruir el archivo OSM a partir del grafo de OSMnx, se utilizó el mismo polígono oficial de Usaquén para realizar una consulta directa a Overpass.

El flujo definitivo adoptado fue:

    Límite oficial de Usaquén
            ↓
    Polígono WGS84
            ↓
    Consulta Overpass
            ↓
    usaquen.osm.xml
            ↓
    netconvert
            ↓
    usaquen.net.xml

De esta manera, la conversión a SUMO utiliza directamente los datos OSM descargados para el área de estudio.

### 6. Generación de la red SUMO

Se generó una primera versión de la red mediante:

    netconvert --osm-files data/processed/usaquen.osm.xml `
      --output-file data/processed/usaquen.net.xml `
      --geometry.remove `
      --roundabouts.guess `
      --ramps.guess `
      --junctions.join `
      --tls.guess-signals `
      --tls.discard-simple

La red resultante fue inspeccionada mediante NetEdit y se verificó visualmente que presenta una estructura conectada y coherente con la red vial de la localidad.

### 7. Consideración de peatones y bicicletas

Debido a que el alcance de la simulación incluye vehículos, bicicletas y peatones, se mantuvieron categorías de infraestructura no motorizada presentes en OSM, entre ellas:

- `footway`
- `cycleway`
- `path`
- `pedestrian`
- `steps`
- `track`
- `corridor`

junto con las categorías vehiculares:

- `trunk`
- `primary`
- `secondary`
- `tertiary`
- `residential`
- `service`
- `unclassified`

Por lo tanto, la extracción ya no se limita exclusivamente a infraestructura destinada al tráfico vehicular motorizado.

## Resultados

Se generaron los siguientes archivos:

    data/processed/usaquen.osm.xml
    data/processed/usaquen.net.xml

El archivo `.net.xml` constituye la primera versión de prueba de la red de simulación SUMO para Usaquén.

La red obtenida conserva infraestructura relevante para el modelamiento multimodal de:

- Vehículos
- Bicicletas
- Peatones

## Hallazgos

- `network_type="all"` resulta más apropiado que un filtro exclusivamente vehicular debido al alcance multimodal definido para la simulación.
- Las categorías `footway`, `cycleway`, `path`, `pedestrian` y `steps` deben ser consideradas para representar adecuadamente los desplazamientos peatonales y ciclistas.
- La simplificación de OSMnx puede generar aristas con múltiples valores de `highway`; este comportamiento se evitó para el flujo definitivo al utilizar directamente los datos OSM descargados mediante Overpass.
- La red obtenida mediante OSMnx fue verificada como conectada, por lo que los segmentos visualmente aislados observados inicialmente no correspondían a componentes desconectados del grafo.
- La utilización directa del archivo OSM descargado mediante Overpass permitió obtener una conversión más adecuada para su posterior utilización en SUMO.
- Se generó y verificó visualmente una primera red `.net.xml` mediante NetEdit.

## Pendientes

1. Analizar los atributos `maxspeed` y `lanes` de la red.
2. Identificar el porcentaje de tramos que no cuentan con estos atributos.
3. Definir y documentar valores por defecto para los atributos faltantes.
4. Revisar las capacidades de las diferentes categorías de vías.
5. Configurar la accesibilidad específica para vehículos, bicicletas y peatones.
6. Revisar la generación de aceras y cruces peatonales.
7. Validar la red multimodal resultante en NetEdit y posteriormente en SUMO.

## Estado

**Primera red `.net.xml` generada y validada visualmente en NetEdit.**